In [ ]:
# Libraries

import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import torch
import torch.nn as nn
import random
from torch.utils.data import Dataset, DataLoader, TensorDataset
from torch_geometric.datasets import TUDataset
from torch_geometric.loader import DataLoader
import wandb

In [ ]:
# Models and Embeddings

# EMBEDDINGS
class GaussianEmbedding(nn.Module):
    def __init__(self, num_terms, num_channels):
        super(GaussianEmbedding, self).__init__()
        self.num_terms = num_terms
        self.num_channels = num_channels

        self.h = nn.Parameter(torch.randn(num_terms + 1, num_channels))
        nn.init.xavier_uniform_(self.h)

    def forward(self, A):
        batch_size, num_nodes, _ = A.shape
        Y_hat = torch.zeros(batch_size, num_nodes, self.num_channels, device=A.device)
        for c in range(self.num_channels):
            result = self.h[0, c] * torch.eye(num_nodes, device=A.device).unsqueeze(
                0
            ).expand(batch_size, -1, -1)
            for i in range(1, self.num_terms + 1):
                A_power_i = torch.matrix_power(A, i)
                result += self.h[i, c] * A_power_i
            Y_hat[..., c] = torch.diagonal(result, dim1=-2, dim2=-1)
        return Y_hat


class EigenEmbedding(nn.Module):
    def __init__(self, num_eigenvectors=None):
        super(EigenEmbedding, self).__init__()
        self.num_eigenvectors = num_eigenvectors

    def forward(self, A):
        eigenvectors = []
        for i in range(A.shape[0]):
            _, V = torch.linalg.eigh(A[i])
            if self.num_eigenvectors is not None:
                eigenvectors.append(V[:, -self.num_eigenvectors :])
            else:
                eigenvectors.append(V)
        return torch.stack(eigenvectors, dim=0)


# MODELS
class GraphConvolutionFilter(nn.Module):
    def __init__(self, K, F, G, layer_norm=False):
        super(GraphConvolutionFilter, self).__init__()

        self.K = K  # number of filter taps
        self.F = F  # number of input channels
        self.G = G  # number of output channels

        self.H = nn.Parameter(torch.empty(K, F, G))
        nn.init.xavier_uniform_(self.H)
        # with torch.no_grad():
        #     for i in range(K):
        #         if i > 1:
        #             self.H[i].data = self.H[i].data * (0.1**i)
        # nn.init.kaiming_uniform_(self.H)

        if layer_norm:
            self.layer_norm = nn.LayerNorm(G)
        else:
            self.layer_norm = None

    def forward(self, A, X):
        # input X is B x N x F
        Z = X @ self.H[0]

        A_power_i = A.clone()

        for i in range(1, self.K):
            Z += torch.bmm(A_power_i, (X @ self.H[i]))
            A_power_i = torch.bmm(A_power_i, A)

        if self.layer_norm is not None:
            Z = self.layer_norm(Z)

        return Z


class LinearPE(nn.Module):
    def __init__(self, F, D, bias=True, layer_norm=False):
        super(LinearPE, self).__init__()

        self.F = F  # number of input channels
        self.D = D  # number of output channels

        self.W = nn.Parameter(torch.randn(F, D))

        if bias:
            self.b = nn.Parameter(torch.randn(D))
        else:
            self.b = torch.zeros(D)

        if layer_norm:
            self.layer_norm = nn.LayerNorm(D)
        else:
            self.layer_norm = None

    def forward(self, A, X):
        # input X is B x N x F

        Z = X @ self.W + self.b

        if self.layer_norm is not None:
            Z = self.layer_norm(Z)

        return Z


class MultiHeadAttention(nn.Module):
    """
    Multi-Head Attention module as described in 'Attention Is All You Need' paper.

    This implementation supports masked attention and different input/output dimensions.
    Optionally, softmax can be skipped in the attention calculation.
    """

    def __init__(
        self,
        d_model,
        num_heads,
        d_k=None,
        d_v=None,
        dropout=0.0,
        bias=False,
        filter_qk=False,
        filter_v=False,
        filter_num_terms=2,
        apply_softmax=True,  # Add option to control softmax application
    ):
        """
        Initialize the Multi-Head Attention module.

        Parameters:
        - d_model: Model dimension (input and output dimension)
        - num_heads: Number of attention heads
        - d_k: Dimension of keys (default: d_model // num_heads)
        - d_v: Dimension of values (default: d_model // num_heads)
        - dropout: Dropout probability
        - apply_softmax: Whether to apply softmax to attention scores
        """
        super(MultiHeadAttention, self).__init__()

        self.num_heads = num_heads
        self.d_model = d_model

        # If d_k and d_v are not specified, set them to d_model // num_heads
        self.d_k = d_k if d_k is not None else d_model // num_heads
        self.d_v = d_v if d_v is not None else d_model // num_heads

        # Linear projections for queries, keys, and values
        if filter_qk:
            # Use a graph convolutional filter layer for Q and K projections
            self.W_q = GraphConvolutionFilter(K=filter_num_terms, F=d_model, G=num_heads * self.d_k, layer_norm=True)
            self.W_k = GraphConvolutionFilter(K=filter_num_terms, F=d_model, G=num_heads * self.d_k, layer_norm=True)
        else:
            self.W_q = nn.Linear(d_model, num_heads * self.d_k, bias=bias)
            self.W_k = nn.Linear(d_model, num_heads * self.d_k, bias=bias)

        if filter_v:
            self.W_v = GraphConvolutionFilter(K=filter_num_terms, F=d_model, G=num_heads * self.d_v)
        else:
            self.W_v = nn.Linear(d_model, num_heads * self.d_v, bias=bias)

        # Output projection
        self.W_o = nn.Linear(num_heads * self.d_v, d_model, bias=bias)

        # Dropout layer
        self.dropout = nn.Dropout(dropout)

        # Layer normalization for the output
        self.layer_norm = nn.LayerNorm(d_model)

        # Scaling factor for dot product attention
        self.scale = 1

        # Linear layer to combine attention scores from different heads
        self.score_combination = nn.Linear(num_heads, 1, bias=False)

        self.filter_qk = filter_qk
        self.filter_v = filter_v
        self.filter_num_terms = filter_num_terms

        self.apply_softmax = apply_softmax  # Save option

    def forward(self, A, x, mask=None, residual=None):
        """
        Forward pass of the Multi-Head Attention module.

        Parameters:
        - Q: Query tensor of shape (batch_size, seq_len_q, d_model)
        - K: Key tensor of shape (batch_size, seq_len_k, d_model)
        - V: Value tensor of shape (batch_size, seq_len_v, d_model)
        - mask: Optional mask tensor of shape (batch_size, seq_len_q, seq_len_k)
        - residual: Optional residual connection

        Returns:
        - output: Output tensor of shape (batch_size, seq_len_q, d_model)
        - attention: Attention weights of shape (batch_size, num_heads, seq_len_q, seq_len_k)
        """

        batch_size = x.size(0)

        # If residual connection is not provided, use Q as residual
        if residual is None:
            residual = x

        # Linear projections and reshaping for multi-head attention
        # Shape: (batch_size, seq_len, num_heads, d_*)
        if self.filter_qk:
            q = self.W_q(A, x)
            k = self.W_k(A, x)
        else:
            q = self.W_q(x)
            k = self.W_k(x)
        if self.filter_v:
            v = self.W_v(A, x)
        else:
            v = self.W_v(x)

        q = q.view(batch_size, -1, self.num_heads, self.d_k)
        k = k.view(batch_size, -1, self.num_heads, self.d_k)
        v = v.view(batch_size, -1, self.num_heads, self.d_v)

        # Transpose to shape: (batch_size, num_heads, seq_len, d_*)
        q = q.transpose(1, 2)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)

        # Calculate attention scores
        # (batch_size, num_heads, seq_len_q, seq_len_k)
        scores = torch.matmul(q, k.transpose(-2, -1)) * self.scale

        # Apply mask if provided
        if mask is not None:
            # Add an extra dimension for the number of heads
            if mask.dim() == 3:  # (batch_size, seq_len_q, seq_len_k)
                mask = mask.unsqueeze(1)
            # Set masked positions to a large negative value before softmax
            scores = scores.masked_fill(mask == 0, -1e9)

        # Optionally apply softmax to get attention weights
        if self.apply_softmax:
            attn_weights = torch.nn.functional.softmax(scores, dim=-1)
        else:
            attn_weights = scores

        # Apply dropout to attention weights (or raw scores if softmax skipped)
        attn_weights = self.dropout(attn_weights)

        # Calculate weighted sum of values
        # (batch_size, num_heads, seq_len_q, d_v)
        context = torch.matmul(attn_weights, v)

        # Transpose and reshape to (batch_size, seq_len_q, num_heads * d_v)
        context = (
            context.transpose(1, 2)
            .contiguous()
            .view(batch_size, -1, self.num_heads * self.d_v)
        )

        # Apply output projection
        output = self.W_o(context)

        # Apply dropout and residual connection
        output = self.dropout(output)
        output = self.layer_norm(output + residual)

        # Combine attention scores from different heads using learned weights
        # Transpose scores to have heads dimension last: (batch_size, seq_len_q, seq_len_k, num_heads)
        scores_for_combination = scores.permute(0, 2, 3, 1)
        # Apply linear combination: (batch_size, seq_len_q, seq_len_k, 1)
        combined_scores = self.score_combination(scores_for_combination)
        # Remove last singleton dimension
        combined_scores = combined_scores.squeeze(-1)

        return output, combined_scores


class MultilayerAttention(nn.Module):
    """
    Stacks multiple MultiHeadAttention layers in sequence.
    Output of layer i is passed as input to layer i+1; A is passed to every layer.
    Each layer uses a residual connection from its input (pre-norm style).
    """

    def __init__(self, n_layers, **mha_kwargs):
        super(MultilayerAttention, self).__init__()
        self.n_layers = n_layers
        self.layers = nn.ModuleList([
            MultiHeadAttention(**mha_kwargs) for _ in range(n_layers)
        ])

    def forward(self, A, x, mask=None):
        """
        Parameters
        ----------
        A : Tensor
            Adjacency matrix (batch_size, n, n), passed to every layer.
        x : Tensor
            Node features (batch_size, n, d_model).
        mask : Tensor, optional
            Attention mask (batch_size, n, n).

        Returns
        -------
        output : Tensor
            Node features after all layers (batch_size, n, d_model).
        combined_scores : Tensor
            Combined attention scores from the last layer (batch_size, n, n).
        """
        out = x
        combined_scores = None
        for layer in self.layers:
            out, combined_scores = layer(A, out, mask=mask, residual=out)
        return out, combined_scores


class SingleHeadQK(nn.Module):
    """
    Single attention head that outputs raw QK^T (attention logits).
    No layer norms, no output projection, no softmax, no MLP.
    Optional: use graph convolution for Q/K (filter_qk=True) or linear projection.
    """

    def __init__(self, d_model, d_k=None, filter_qk=False, filter_num_terms=2):
        super(SingleHeadQK, self).__init__()
        self.d_model = d_model
        self.d_k = d_k if d_k is not None else d_model
        self.filter_qk = filter_qk
        if filter_qk:
            self.W_q = GraphConvolutionFilter(
                K=filter_num_terms, F=d_model, G=self.d_k
            )
            self.W_k = GraphConvolutionFilter(
                K=filter_num_terms, F=d_model, G=self.d_k
            )
        else:
            self.W_q = nn.Linear(d_model, self.d_k, bias=False)
            self.W_k = nn.Linear(d_model, self.d_k, bias=False)
        self.scale = self.d_k ** (-0.5)

    def forward(self, A, x, mask=None):
        """
        Parameters
        ----------
        A : Tensor
            Adjacency (batch_size, n, n). Used only if filter_qk=True.
        x : Tensor
            Node features (batch_size, n, d_model).
        mask : Tensor, optional
            (batch_size, n, n). Masked positions get -inf before return.

        Returns
        -------
        scores : Tensor
            QK^T of shape (batch_size, n, n), scaled by 1/sqrt(d_k).
        """
        if self.filter_qk:
            q = self.W_q(A, x)  # (batch_size, n, d_k)
            k = self.W_k(A, x)  # (batch_size, n, d_k)
        else:
            q = self.W_q(x)
            k = self.W_k(x)
        scores = torch.matmul(q, k.transpose(-2, -1)) * self.scale  # (batch_size, n, n)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        return None, scores


    

In [ ]:
# Define sequential model combining embedding and denoising
class SequentialDenoisingModel(nn.Module):
    def __init__(
        self,
        embedding_model,
        denoising_model,
    ):
        super(SequentialDenoisingModel, self).__init__()
        self.embedding_model = embedding_model
        self.denoising_model = denoising_model

    def forward(self, A):
        # First apply embedding model to get node embeddings
        E = self.embedding_model(A)

        # Then apply denoising model to get denoised embeddings
        Z, A_recon = self.denoising_model(A, E.float())

        return A_recon

In [ ]:
# Assorted functions

def generate_sbm_adjacency(block_sizes, p, q, rng=None):
    """
    Generate an adjacency matrix for a stochastic block model with variable block sizes.

    Parameters:
    - block_sizes: List of sizes for each block.
    - p: Probability of intra-block edges.
    - q: Probability of inter-block edges.
    - rng: Random number generator (optional).

    Returns:
    - Adjacency matrix as a numpy array.
    """
    if rng is None:
        rng = np.random.default_rng()

    n_blocks = len(block_sizes)
    n = sum(block_sizes)

    # Initialize the adjacency matrix with zeros

    adj_matrix = np.zeros((n, n))

    # Calculate the starting index of each block
    block_starts = [0]
    for i in range(n_blocks - 1):
        block_starts.append(block_starts[-1] + block_sizes[i])

    for i in range(n_blocks):
        for j in range(i, n_blocks):
            density = p if i == j else q
            block_start_i = block_starts[i]
            block_end_i = block_start_i + block_sizes[i]
            block_start_j = block_starts[j]
            block_end_j = block_start_j + block_sizes[j]

            # Generate random edges within or between blocks
            block_i_size = block_sizes[i]
            block_j_size = block_sizes[j]
            adj_matrix[block_start_i:block_end_i, block_start_j:block_end_j] = (
                rng.random((block_i_size, block_j_size)) < density
            ).astype(int)

            # Make the matrix symmetric (for undirected graphs)
            if i != j:
                adj_matrix[block_start_j:block_end_j, block_start_i:block_end_i] = (
                    adj_matrix[block_start_i:block_end_i, block_start_j:block_end_j].T
                )

    return adj_matrix


def add_digress_noise(A, p, rng=None, symmetric=True):
    """
    Add noise to an adjacency matrix or batch by flipping edges with probability p.

    Parameters:
    - A: 2D or 3D numpy array or tensor representing an adjacency matrix (0s and 1s)
         (n, n) or (bs, n, n)
    - p: Probability of flipping each element (0 to 1, 1 becomes 0 and 0 becomes 1)
    - rng: Random number generator (optional)
    - symmetric: If True, noise pattern will be symmetric (upper-tri noise mirrored to lower-tri)

    Returns:
    - Tuple: (Noisy adj, V, eigenvalues), all torch.float32 tensors with batch dim if input has batch
    """
    if rng is None:
        rng = np.random.default_rng()

    # Accept numpy arrays or tensors, convert to tensor if needed
    if not torch.is_tensor(A):
        A_tensor = torch.tensor(A)
    else:
        A_tensor = A

    device = A_tensor.device if hasattr(A_tensor, "device") else "cpu"
    dtype = A_tensor.dtype

    # Detect batch: 2D (n, n) or 3D (bs, n, n)
    if A_tensor.dim() == 2:
        # single matrix: (n, n)
        n = A_tensor.shape[0]
        batch_shape = None
        input_was_batched = False
    elif A_tensor.dim() == 3:
        # batch: (bs, n, n)
        bs, n, n2 = A_tensor.shape
        assert n == n2, "Input tensor shape must be (bs, n, n)"
        batch_shape = (bs,)
        input_was_batched = True
    else:
        raise ValueError("Input adjacency must have shape (n, n) or (bs, n, n)")

    if symmetric:
        # Create upper-triangle mask for single or batch input
        if batch_shape is None:
            # (n, n)
            random_values_upper = torch.rand((n, n), device=device)
            upper_tri_mask = torch.triu(torch.ones((n, n), device=device, dtype=torch.bool))
            random_values_upper[~upper_tri_mask] = 1.1  # set below-tri to >p so never flips
            flip_mask_upper = (random_values_upper < p)
            # Mirror upper to lower for symmetry
            flip_mask = flip_mask_upper | flip_mask_upper.t()
        else:
            # (bs, n, n)
            random_values_upper = torch.rand((bs, n, n), device=device)
            # Broadcast upper triangle mask to batch
            upper_tri_mask = torch.triu(torch.ones((n, n), device=device, dtype=torch.bool)).unsqueeze(0)  # shape (1, n, n)
            upper_tri_mask = upper_tri_mask.expand(bs, n, n)
            random_values_upper[~upper_tri_mask] = 1.1  # only upper triangle can flip
            flip_mask_upper = (random_values_upper < p)
            # Mirror upper to lower along last two dims
            flip_mask = flip_mask_upper | flip_mask_upper.transpose(1, 2)
    else:
        if batch_shape is None:
            random_values = torch.rand_like(A_tensor)
            flip_mask = random_values < p
        else:
            random_values = torch.rand((bs, n, n), device=device)
            flip_mask = random_values < p

    # Flip edges according to mask (works for 2D or 3D)
    A_noisy = torch.where(flip_mask, 1 - A_tensor, A_tensor)

    # eigendecomp batch or not
    if A_noisy.dim() == 2:
        eigvals, V = torch.linalg.eigh(A_noisy.float())
    else:
        # Batched eigendecomposition
        # torch.linalg.eigh supports batch mode
        eigvals, V = torch.linalg.eigh(A_noisy.float())

    # Always return with batch if input had batch
    return (
        torch.tensor(A_noisy, dtype=torch.float32),
        torch.tensor(V, dtype=torch.float32),
        torch.tensor(eigvals, dtype=torch.float32),
    )


def generate_block_sizes(n, min_blocks=2, max_blocks=6, min_size=2, max_size=20):
    # Example usage:
    # n is the number of nodes
    # n = 20
    # partitions = generate_block_sizes(n)
    # print(f"Valid block size partitions for n={n}:")
    # for p in partitions:
    #     print(p)
    valid_partitions = []

    # Try different numbers of blocks
    for num_blocks in range(min_blocks, max_blocks + 1):

        def generate_partitions(remaining, blocks_left, current_partition):
            # Base cases
            if blocks_left == 0:
                if remaining == 0:
                    valid_partitions.append(current_partition[:])
                return

            # Try different sizes for current block
            start = max(min_size, remaining - (blocks_left - 1) * max_size)
            end = min(max_size, remaining - (blocks_left - 1) * min_size) + 1

            for size in range(start, end):
                if size <= remaining:
                    current_partition.append(size)
                    generate_partitions(
                        remaining - size, blocks_left - 1, current_partition
                    )
                    current_partition.pop()

        generate_partitions(n, num_blocks, [])

    return valid_partitions


class PermutedAdjacencyDataset(Dataset):
    def __init__(self, adjacency_matrices, num_samples):
        self.adjacency_matrices = adjacency_matrices
        self.num_samples = num_samples

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        # Randomly choose between A_1 and A_2
        matrix_idx = torch.randint(0, len(self.adjacency_matrices), (1,)).item()
        adjacency_matrix = self.adjacency_matrices[matrix_idx]

        # Generate random permutation
        permuted_indices = torch.randperm(adjacency_matrix.size(0))
        A_permuted = adjacency_matrix[permuted_indices, :][:, permuted_indices]
        return A_permuted
        

In [ ]:
# PyTorch Geometric dataset wrapper for TU datasets (adjacency-only).

# This module provides a wrapper around PyTorch Geometric TUDatasets that
# extracts adjacency matrices and handles variable graph sizes via padding.
# Node features are ignored by design.


from pathlib import Path

import numpy as np
import torch


class TUDatasetWrapper:
    """Wrapper for PyTorch Geometric TU datasets extracting adjacency matrices.

    Handles variable-size graphs by padding all adjacency matrices to the
    maximum graph size in the dataset.

    Parameters
    ----------
    dataset_name : str
        Name of the TU dataset (e.g., "ENZYMES", "PROTEINS", "MUTAG").
    root : str or Path, optional
        Root directory for downloading/storing datasets.
        Default is "~/.pyg_data".
    max_graphs : int, optional
        Maximum number of graphs to load. Default is None (all graphs).

    Attributes
    ----------
    adjacencies : np.ndarray
        Padded adjacency matrices of shape (num_graphs, max_n, max_n).
    num_nodes : np.ndarray
        Actual number of nodes per graph (before padding).
    max_n : int
        Maximum number of nodes across all graphs.
    """

    def __init__(
        self,
        dataset_name: str,
        root: str | None = None,
        max_graphs: int | None = None,
    ):
        self.dataset_name = dataset_name
        self.root = Path(root) if root else Path.home() / ".pyg_data"
        self.max_graphs = max_graphs

        self._load_dataset()

    def _load_dataset(self) -> None:
        """Load TU dataset and extract padded adjacency matrices."""
        try:
            from torch_geometric.datasets import TUDataset
            from torch_geometric.utils import to_dense_adj
        except ImportError as e:
            raise ImportError(
                "PyTorch Geometric is required. "
                "Install with: pip install torch-geometric"
            ) from e

        # Load TU dataset
        dataset = TUDataset(
            root=str(self.root),
            name=self.dataset_name,
        )

        if self.max_graphs is not None:
            dataset = dataset[: self.max_graphs]

        adjacencies = []
        num_nodes = []

        for data in dataset:
            n = int(data.num_nodes)
            num_nodes.append(n)

            A = to_dense_adj(
                data.edge_index,
                max_num_nodes=n,
            ).squeeze(0)

            adjacencies.append(A.cpu().numpy().astype(np.float32))

        self.num_nodes = np.asarray(num_nodes, dtype=np.int64)
        self.max_n = int(self.num_nodes.max())
        self._num_graphs = len(adjacencies)

        # Pad adjacency matrices
        padded = np.zeros(
            (self._num_graphs, self.max_n, self.max_n),
            dtype=np.float32,
        )

        for i, A in enumerate(adjacencies):
            n = A.shape[0]
            padded[i, :n, :n] = A

        self.adjacencies = padded

    def __len__(self) -> int:
        return self._num_graphs

    def __getitem__(self, idx: int) -> np.ndarray:
        return self.adjacencies[idx]

    def get_adjacency_matrices(self) -> np.ndarray:
        """Return all padded adjacency matrices."""
        return self.adjacencies

    def get_node_counts(self) -> np.ndarray:
        """Return number of nodes per graph (before padding)."""
        return self.num_nodes

    def to_torch(self) -> torch.Tensor:
        """Return adjacency matrices as a float32 torch tensor."""
        return torch.from_numpy(self.adjacencies).float()

    def get_masks(self) -> torch.Tensor:
        """Return node masks for valid (non-padded) entries.

        Returns
        -------
        torch.Tensor
            Boolean tensor of shape (num_graphs, max_n, max_n).
        """
        masks = torch.zeros(
            self._num_graphs, self.max_n, self.max_n, dtype=torch.bool
        )
        for i, n in enumerate(self.num_nodes):
            masks[i, :n, :n] = True
        return masks

    def train_val_test_split(
        self,
        train_ratio: float = 0.7,
        val_ratio: float = 0.1,
        seed: int | None = None,
    ) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
        """Random train/val/test split."""
        rng = np.random.default_rng(seed)
        indices = rng.permutation(self._num_graphs)

        n_train = int(train_ratio * self._num_graphs)
        n_val = int(val_ratio * self._num_graphs)

        train_idx = indices[:n_train]
        val_idx = indices[n_train : n_train + n_val]
        test_idx = indices[n_train + n_val :]

        return (
            self.adjacencies[train_idx],
            self.adjacencies[val_idx],
            self.adjacencies[test_idx],
        )


def load_tu_dataset(
    name: str,
    root: str | None = None,
    max_graphs: int | None = None,
) -> TUDatasetWrapper:
    """Convenience loader for TU datasets.

    Examples
    --------
    >>> dataset = load_tu_dataset("ENZYMES", max_graphs=100)
    >>> dataset.max_n
    """
    return TUDatasetWrapper(
        dataset_name=name,
        root=root,
        max_graphs=max_graphs,
    )



In [ ]:
# PyTorch Geometric dataset wrappers for graph denoising experiments.

# This module provides wrappers around PyTorch Geometric datasets (QM9, ENZYMES,
# PROTEINS) that extract adjacency matrices and handle variable graph sizes
# through padding.


from pathlib import Path

import numpy as np
import torch


class PyGDatasetWrapper:
    """Wrapper for PyTorch Geometric datasets that extracts adjacency matrices.

    Handles variable-size graphs by padding to the maximum size in the dataset.
    Node features are ignored as per experiment design (adjacency-only).

    Parameters
    ----------
    dataset_name : str
        Name of the dataset: "qm9", "enzymes", or "proteins".
    root : str or Path, optional
        Root directory for downloading/storing datasets.
        Default is "~/.pyg_data".
    max_graphs : int, optional
        Maximum number of graphs to load. Default is None (all graphs).

    Attributes
    ----------
    adjacencies : np.ndarray
        Padded adjacency matrices of shape (num_graphs, max_n, max_n).
    num_nodes : np.ndarray
        Actual number of nodes per graph (before padding).
    max_n : int
        Maximum number of nodes across all graphs.
    """

    VALID_DATASETS = {"qm9", "enzymes", "proteins"}

    def __init__(
        self,
        dataset_name: str,
        root: str | None = None,
        max_graphs: int | None = None,
    ):
        if dataset_name.lower() not in self.VALID_DATASETS:
            raise ValueError(
                f"dataset_name must be one of {self.VALID_DATASETS}, "
                f"got '{dataset_name}'"
            )

        self.dataset_name = dataset_name.lower()
        self.root = Path(root) if root else Path.home() / ".pyg_data"
        self.max_graphs = max_graphs

        # Load dataset
        self._load_dataset()

    def _load_dataset(self) -> None:
        """Load PyG dataset and extract adjacency matrices."""
        try:
            from torch_geometric.datasets import QM9, TUDataset
            from torch_geometric.utils import to_dense_adj
        except ImportError as e:
            raise ImportError(
                "PyTorch Geometric is required for benchmark datasets. "
                "Install with: pip install torch-geometric"
            ) from e

        # Load appropriate dataset
        if self.dataset_name == "qm9":
            dataset = QM9(root=str(self.root / "QM9"))
        elif self.dataset_name == "enzymes":
            dataset = TUDataset(root=str(self.root), name="ENZYMES")
        elif self.dataset_name == "proteins":
            dataset = TUDataset(root=str(self.root), name="PROTEINS")
        else:
            raise ValueError(f"Unknown dataset: {self.dataset_name}")

        # Limit number of graphs if specified
        if self.max_graphs is not None:
            dataset = dataset[: self.max_graphs]

        # Extract adjacency matrices
        adjacencies = []
        num_nodes = []

        for i in range(len(dataset)):
            data = dataset[i]  # pyright: ignore[reportArgumentType]
            # Get number of nodes - use getattr for PyG Data dynamic attributes
            n = getattr(data, "num_nodes")  # noqa: B009
            num_nodes.append(n)

            # Convert edge_index to dense adjacency
            edge_index = getattr(data, "edge_index")  # noqa: B009
            A = to_dense_adj(edge_index, max_num_nodes=n).squeeze(0)
            adjacencies.append(A.numpy().astype(np.float32))

        self.num_nodes = np.array(num_nodes)
        self.max_n = int(self.num_nodes.max())

        # Pad all adjacencies to max size
        padded = []
        for A in adjacencies:
            n = A.shape[0]
            if n < self.max_n:
                # Pad with zeros
                padded_A = np.zeros((self.max_n, self.max_n), dtype=np.float32)
                padded_A[:n, :n] = A
                padded.append(padded_A)
            else:
                padded.append(A)

        self.adjacencies = np.stack(padded, axis=0)
        self._num_graphs = len(self.adjacencies)

    def __len__(self) -> int:
        return self._num_graphs

    def __getitem__(self, idx: int) -> np.ndarray:
        return self.adjacencies[idx]

    def get_adjacency_matrices(self) -> np.ndarray:
        """Return all padded adjacency matrices.

        Returns
        -------
        np.ndarray
            Adjacency matrices of shape (num_graphs, max_n, max_n).
        """
        return self.adjacencies

    def get_node_counts(self) -> np.ndarray:
        """Return actual node counts for each graph.

        Returns
        -------
        np.ndarray
            Number of nodes per graph (before padding).
        """
        return self.num_nodes

    def to_torch(self) -> torch.Tensor:
        """Convert adjacency matrices to PyTorch tensor.

        Returns
        -------
        torch.Tensor
            Adjacency matrices as float32 tensor.
        """
        return torch.from_numpy(self.adjacencies).float()

    def get_masks(self) -> torch.Tensor:
        """Get node masks indicating valid (non-padded) regions.

        Returns
        -------
        torch.Tensor
            Boolean masks of shape (num_graphs, max_n) where True indicates
            a valid node.
        """
        masks = torch.zeros(self._num_graphs, self.max_n, self.max_n, dtype=torch.bool)
        for i, n in enumerate(self.num_nodes):
            masks[i, :n, :n] = True
        return masks

    def train_val_test_split(
        self,
        train_ratio: float = 0.7,
        val_ratio: float = 0.1,
        seed: int | None = None,
    ) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
        """Split dataset into train/validation/test sets.

        Parameters
        ----------
        train_ratio : float
            Fraction of data for training. Default 0.7.
        val_ratio : float
            Fraction of data for validation. Default 0.1.
        seed : int, optional
            Random seed for shuffling.

        Returns
        -------
        train : np.ndarray
            Training adjacency matrices.
        val : np.ndarray
            Validation adjacency matrices.
        test : np.ndarray
            Test adjacency matrices.
        """
        rng = np.random.default_rng(seed)
        indices = rng.permutation(self._num_graphs)

        n_train = int(train_ratio * self._num_graphs)
        n_val = int(val_ratio * self._num_graphs)

        train_idx = indices[:n_train]
        val_idx = indices[n_train : n_train + n_val]
        test_idx = indices[n_train + n_val :]

        return (
            self.adjacencies[train_idx],
            self.adjacencies[val_idx],
            self.adjacencies[test_idx],
        )


def load_pyg_dataset(
    name: str,
    root: str | None = None,
    max_graphs: int | None = None,
) -> PyGDatasetWrapper:
    """Convenience function to load a PyG dataset.

    Parameters
    ----------
    name : str
        Dataset name: "qm9", "enzymes", or "proteins".
    root : str, optional
        Root directory for datasets.
    max_graphs : int, optional
        Maximum number of graphs to load.

    Returns
    -------
    PyGDatasetWrapper
        Loaded dataset wrapper.

    Examples
    --------
    >>> dataset = load_pyg_dataset("enzymes", max_graphs=100)
    >>> print(f"Loaded {len(dataset)} graphs with max {dataset.max_n} nodes")
    """
    return PyGDatasetWrapper(name, root=root, max_graphs=max_graphs)

from torch.utils.data import Dataset, DataLoader

class PygAdjacencyDataset(Dataset):
    def __init__(self, adj_mats, masks):
        self.adj_mats = adj_mats
        self.masks = masks

    def __len__(self):
        return self.adj_mats.shape[0]

    def __getitem__(self, idx):
        # Return adjacency and corresponding mask
        return (
            torch.tensor(self.adj_mats[idx], dtype=torch.float32),
            self.masks[idx],
        )


In [ ]:
# Training Loop

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print('device:', device)




seed = 42

random.seed(seed)  # For reproducibility
torch.manual_seed(seed)
np.random.seed(seed)

# hyperparameters
#######################################


# number of nodes
n = 30



num_samples = 128  # number of permutations

test_epochs = 10 #every how many epochs to evaluate the model and visualize the results
train_batch_size = 64
test_batch_size = 64
dropout = 0.0


dataset_name = "sbm"
num_partitions = 5
max_graphs = 50


    
    
use_wandb = False
wandb_project = "graph-denoising-final-3"
wandb_entity = "graph_denoise_team"


level = 0.35
noise_levels = [level, level] #could be a list of different noise levels (at least 2)

use_eigenvectors = True
num_eigenvectors = 10
H = 1 # number of heads in the transformer
K = 3  # number of filter taps in the graph filter banks
F = 50
G = 10

num_epochs = 200

loss_type = "MSE"  # loss criteria: either "MSE" or "BCEwithLogits"
learning_rate = 0.01

########################################
#CREATING DATASET

if dataset_name == "sbm":
    

    p_intra = 1.0
    q_inter = 0.0

    # gives all possible partitions of n into 2-4 blocks
    partitions = generate_block_sizes(n, min_blocks=2, max_blocks=5, min_size=5, max_size=25)

    # Sample num_partitions many partitions for training and num_partitions many partitions for test
    train_partitions = random.sample(partitions, num_partitions)
    test_partitions = random.sample(
        [p for p in partitions if p not in train_partitions], 2*num_partitions
    )

    # train_partitions = [[20, 40]]
    # test_partitions = [[10, 50]]
    As_train = []
    As_test = []

    for p in train_partitions:
        A = generate_sbm_adjacency(p, p_intra, q_inter)
        A = torch.tensor(A, dtype=torch.float)
        As_train.append(A)

    for p in test_partitions:
        A = generate_sbm_adjacency(p, p_intra, q_inter)
        A = torch.tensor(A, dtype=torch.float)
        As_test.append(A)

    print("\nTraining partitions:")
    for p in train_partitions:
        print(p)

    print("\nTest partitions:")
    for p in test_partitions:
        print(p)

    train_dataset = PermutedAdjacencyDataset(As_train, num_samples)
    train_dataloader = DataLoader(train_dataset, batch_size=train_batch_size, shuffle=True)

    test_dataset = PermutedAdjacencyDataset(As_test, num_samples)
    test_dataloader = DataLoader(test_dataset, batch_size=test_batch_size, shuffle=True)


if dataset_name != "sbm":
    #Shervin: datasets that I have tried other than sbm
    if dataset_name == "pyg_enzymes":
        dataset = load_pyg_dataset("enzymes", max_graphs=max_graphs)
    elif dataset_name == "pyg_qm9":
        dataset = load_pyg_dataset("qm9", max_graphs=max_graphs)
    elif dataset_name == "pyg_proteins":
        dataset = load_pyg_dataset("proteins", max_graphs=max_graphs)
    elif dataset_name == "tu_imdb":
        dataset = load_tu_dataset("IMDB-BINARY", max_graphs=max_graphs)
    elif dataset_name == "tu_deezer_ego_nets":
        dataset = load_tu_dataset("deezer_ego_nets", max_graphs=max_graphs)
    elif dataset_name == "tu_collab":
        dataset = load_tu_dataset("COLLAB", max_graphs=max_graphs)
    
        

    # Get adjacency matrices and node counts, split into train/test sets
    adj_mats = dataset.get_adjacency_matrices()           
    masks = dataset.get_masks()

    from torch.utils.data import Dataset, DataLoader

    class PygAdjacencyDataset(Dataset):
        def __init__(self, adj_mats, masks):
            self.adj_mats = adj_mats
            self.masks = masks

        def __len__(self):
            return self.adj_mats.shape[0]

        def __getitem__(self, idx):
            # Return adjacency and corresponding mask
            return (
                torch.tensor(self.adj_mats[idx], dtype=torch.float32),
                self.masks[idx],
            )

    # Do a train/test split
    num_graphs = adj_mats.shape[0]
    indices = torch.randperm(num_graphs)
    train_ratio = 0.7
    split = int(train_ratio * num_graphs)

    train_idx = indices[:split]
    test_idx = indices[split:]

    train_adjs = adj_mats[train_idx]
    train_masks = masks[train_idx]

    test_adjs = adj_mats[test_idx]
    test_masks = masks[test_idx]

    train_dataset = PygAdjacencyDataset(train_adjs, train_masks)
    test_dataset = PygAdjacencyDataset(test_adjs, test_masks)

    train_dataloader = DataLoader(train_dataset, batch_size=8, shuffle=True)
    test_dataloader = DataLoader(test_dataset, batch_size=8, shuffle=False)

    for batch in train_dataloader:      
        batch_adj, batch_mask = batch
        print("Train batch adjacency shape:", batch_adj.shape)
        print("Train batch mask shape:", batch_mask.shape)
        
        break

    for batch in test_dataloader:
        batch_adj, batch_mask = batch
        print("Test batch adjacency shape:", batch_adj.shape)
        print("Test batch mask shape:", batch_mask.shape)
        break





for embedding_type in ["eigen"]:
    for model_type in ["SelfAttention"]:

    
        print("embedding_type: ", embedding_type)
        print("model_type: ", model_type)

        if use_wandb:
            wandb.init(project=wandb_project, entity=wandb_entity, name=f"{model_type}_dataset{dataset_name}_noise{noise_levels[0]}_loss{loss_type}")

        
        if model_type == "SelfAttention":
            model_denoiser = MultiHeadAttention(
                        d_model=num_eigenvectors, num_heads=H, d_k=G, d_v=G, apply_softmax=False
                    )
        
        elif model_type == "FilterAttention":
            model_denoiser = MultiHeadAttention(
                        d_model=num_eigenvectors, num_heads=H, d_k=G, d_v=G, apply_softmax=False, filter_qk = True, filter_num_terms = K
                    )
        
        elif model_type == "SimpleAttentionFilter":
            model_denoiser = SingleHeadQK(
                        d_model=num_eigenvectors, d_k=G, filter_qk = True
                    )
        elif model_type == "SimpleAttention":
            model_denoiser = SingleHeadQK(
                        d_model=num_eigenvectors, d_k=G, filter_qk = False
                    )

        if embedding_type == "eigen":
            model_embedding = EigenEmbedding(
                        num_eigenvectors=num_eigenvectors
                    )
        elif embedding_type == "gaussian":
            model_embedding = GaussianEmbedding(num_terms=K, num_channels=num_eigenvectors)

        model = SequentialDenoisingModel(
                        model_embedding,
                        model_denoiser,
                    )

        # Calculate and print the number of trainable parameters in the model
        num_trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"Number of trainable parameters of {model_type}: {num_trainable_params}")

        

        model = model.to(device)

        # Log model hyperparameters if using wandb
        if use_wandb:
            wandb.config.update(
                {
                    "loss_type": loss_type,
                    "train_batch_size": train_batch_size,
                    "test_batch_size": test_batch_size,
                    "learning_rate": learning_rate,
                    "embedding_type": embedding_type,
                    "dataset_name": dataset_name,
                    "num_samples": num_samples,
                    "num_epochs": num_epochs,
                    "dropout": dropout,
                    "layer_norm": layer_norm,
                    "final_nonlinearity": final_nonlinearity,
                    "asymmetric": asymmetric,
                    "num_eigenvectors": num_eigenvectors,
                    "K": K,
                    "F": F,
                    "G": G,
                    "num_heads": H,
                    "model_type": model_type,
                    "seed": seed,
                    "num_trainable_params": num_trainable_params,
                    "use_film": use_film,
                }
            )

        optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
            optimizer, T_0=num_epochs//2, T_mult=2
        )
        # scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

        if loss_type == "MSE":
            criterion = nn.MSELoss()
        elif loss_type == "BCE":
            criterion = nn.BCEWithLogitsLoss()

        # Training loop
        for epoch in tqdm(range(num_epochs), desc="Training epochs"):
            model.train()
            epoch_loss = 0.0
            epoch_loss_pred = 0.0
            epoch_loss_baseline = 0.0
            num_batches = 0


            for batch in train_dataloader:
                
                if dataset_name != "sbm":
                    batch, batch_mask = batch
                else:
                    batch_mask = torch.ones_like(batch)


                optimizer.zero_grad()

                # Sample random noise level for this batch
                eps = np.random.choice(noise_levels)
                batch_noisy = (add_digress_noise(batch, eps))[0]

                batch_noisy = batch_noisy.float().to(device)
                batch = batch.float().to(device)


                if dataset_name != "sbm":
                    output = model(batch_noisy*batch_mask.to(device))
                else:
                    output = model(batch_noisy)

                # assert False

                # print(output)
                # output_pred = (torch.nn.functional.sigmoid(output) > 0.5).float()
                if 'digress' in model_type:
                    output_pred = torch.nn.functional.sigmoid(output).float()
                else:
                    output_pred = output.float()

                # output_pred = (output_pred > 0.5).float()

                
                
        

                
            

                # Compute loss
                if dataset_name != "sbm":
                    loss = criterion(output*batch_mask.to(device), batch*batch_mask.to(device))
                else:
                    loss = criterion(output, batch)


                
                epoch_loss += loss.item()
                num_batches += 1

                # Backward pass and optimization
                loss.backward()
                optimizer.step()

                loss_pred = (torch.sum((batch - output_pred)**2*batch_mask.to(device), dim=tuple(range(1, batch_mask.dim())))/batch_mask.sum(dim=tuple(range(1, batch_mask.dim()))).to(device)).detach()
                epoch_loss_pred += loss_pred.mean().item()
                
                loss_baseline = (torch.sum((batch_noisy - batch)**2*batch_mask.to(device), dim=tuple(range(1, batch_mask.dim())))/batch_mask.sum(dim=tuple(range(1, batch_mask.dim()))).to(device)).detach()
                epoch_loss_baseline += loss_baseline.mean().item()

            # Step the scheduler
            scheduler.step()

            # Calculate average epoch loss
            avg_epoch_loss = epoch_loss / num_batches
            avg_epoch_loss_pred = epoch_loss_pred / num_batches
            avg_epoch_loss_baseline = epoch_loss_baseline / num_batches

            # evaluate test error
            test_loss = 0.0
            test_loss_pred = 0.0
            test_loss_baseline = 0.0
            num_batches = 0
            model.eval()
            with torch.no_grad():
                for batch in test_dataloader:
                    if dataset_name != "sbm":
                        batch, batch_mask = batch
                    else:
                        batch_mask = torch.ones_like(batch)
                    
                    # Sample random noise level for this batch
                    eps = np.random.choice(noise_levels)
                    batch_noisy = (add_digress_noise(batch, eps))[0]

                    batch_noisy = batch_noisy.float().to(device)
                    batch = batch.float().to(device)

                    output = model(batch_noisy)

                    # output_pred = (torch.nn.functional.sigmoid(output) > 0.5).float()
                    if 'digress' in model_type:
                        output_pred = torch.nn.functional.sigmoid(output).float()
                    else:
                        output_pred = output.float()

                    # output_pred = (output_pred > 0.5).float()

                    

                    loss_pred = (torch.sum((batch - output_pred)**2*batch_mask.to(device), dim=tuple(range(1, batch_mask.dim())))/batch_mask.sum(dim=tuple(range(1, batch_mask.dim()))).to(device)).detach()
                    test_loss_pred += loss_pred.mean().item()
                    
                    loss_baseline = (torch.sum((batch_noisy - batch)**2*batch_mask.to(device), dim=tuple(range(1, batch_mask.dim())))/batch_mask.sum(dim=tuple(range(1, batch_mask.dim()))).to(device)).detach()
                    test_loss_baseline += loss_baseline.mean().item()
                    

                    # Compute loss
                    loss = criterion(output, batch)
                    test_loss += loss.item()
                    num_batches += 1
            avg_test_loss = test_loss / num_batches
            avg_test_loss_pred = test_loss_pred / num_batches
            avg_test_loss_baseline = test_loss_baseline / num_batches
            # Log metrics to wandb if enabled
            if use_wandb:
                wandb.log(
                    {
                        "train_loss": avg_epoch_loss,
                        "train_loss_pred": avg_epoch_loss_pred,
                        "train_loss_baseline": avg_epoch_loss_baseline,
                        "test_loss": avg_test_loss,
                        "test_loss_pred": avg_test_loss_pred,
                        "test_loss_baseline": avg_test_loss_baseline,
                        "noise_level": eps,
                        "learning_rate": scheduler.get_last_lr()[0],
                    },
                    step=epoch,
                )

            if (epoch) % test_epochs == 0:
                if use_wandb:
                    # evaluate train and test error per noise level
                    model.eval()
                    with torch.no_grad():
                        train_losses = []
                        test_losses = []
                        for eps in noise_levels:
                            train_loss = 0.0
                            test_loss = 0.0
                            num_batches = 0

                            for batch in train_dataloader:
                                if dataset_name != "sbm":
                                    batch, batch_mask = batch
                                else:
                                    batch_mask = torch.ones_like(batch)
                                

                                batch_noisy = (
                                    add_digress_noise(batch, eps)
                                )[0]
                                batch_noisy = batch_noisy.float().to(device)
                                batch = batch.float().to(device)

                                if dataset_name != "sbm":
                                    output = model(batch_noisy*batch_mask.to(device))
                                else:
                                    output = model(batch_noisy)

                                loss = criterion(output, batch)
                                train_loss += loss.item()
                                num_batches += 1

                            train_losses.append(train_loss / num_batches)

                            for batch in test_dataloader:
                                if dataset_name != "sbm":
                                    batch, batch_mask = batch
                                
                                batch_noisy = (
                                    add_digress_noise(batch, eps)
                                )[0]
                                batch_noisy = batch_noisy.float().to(device)
                                batch = batch.float().to(device)
                                output = model(batch_noisy)
                                loss = criterion(output, batch)
                                test_loss += loss.item()
                                num_batches += 1

                            test_losses.append(test_loss / num_batches)

                            wandb.log(
                                {
                                    "eps_"
                                    + str(eps)
                                    + "_train_loss": train_losses[-1],
                                    "eps_"
                                    + str(eps)
                                    + "_test_loss": test_losses[-1],
                                },
                                step=epoch,
                            )

                        # visualize the results
                        eps_values = noise_levels
            
                        # Select random graphs from the test and train sets
                        if dataset_name != "sbm":
                            A_test, A_test_mask = test_dataset[np.random.randint(len(test_dataset))]
                            A_train, A_train_mask = train_dataset[np.random.randint(len(train_dataset))]
                
                    



                            # Clean up selection and mask unpacking logic
                            graph_names = ["Test", "Train"]
                            graph_data = [
                                (A_test, A_test_mask),
                                (A_train, A_train_mask)
                            ]
                            for j, (A_mat, A_mask) in enumerate(graph_data):
                                # Ensure shape/batch dimension
                                if A_mat.ndim == 2:
                                    A_mat = A_mat.unsqueeze(0)
                                # For .unsqueeze(0) on the mask, check first if it's not None and shape should match A_mat
                                if A_mask is not None and A_mask.ndim == 1:
                                    A_mask = A_mask.unsqueeze(0)

                                A_orig = A_mat.to(device)

                                # l_orig, V_orig = torch.linalg.eigh(A_orig)

                                fig_size = 4

                                # Create a figure with subplots - 2 rows for noisy/recon pairs, 5 columns for noise levels
                                fig, axes = plt.subplots(
                                    2,
                                    len(eps_values),
                                    figsize=(
                                        fig_size * len(eps_values),
                                        fig_size * 2,
                                    ),
                                )

                                # Plot pairs of noisy and reconstructed matrices for each noise level
                                for i, eps in enumerate(eps_values):
                                    A_noisy = add_digress_noise(A_orig, eps)[0]
                                    A_recon = model(
                                        A_noisy.float().to(device)
                                    )

                                
                                    # Only visualize the submatrix determined by the mask

                                    # Compute the mask array (assumed to be a [n_total, n_total] matrix with an n x n block of 1s in the top-left)
                                    mask_arr = (A_mask.to(device).squeeze(0) != 0).detach().cpu().numpy()

                                    # Find n: the largest integer such that the top-left n x n block is all True and the rest is zero.
                                    # This works if the mask is a matrix of 1s in the top-left block.
                                    row_sums = mask_arr.sum(axis=1)
                                    n = int((row_sums > 0).sum())

                                    # Prepare noisy and reconstructed matrices
                                    A_noisy_vis = (
                                        A_noisy.to(device).squeeze(0).detach().cpu().numpy()
                                    )
                                    A_recon_vis = (
                                        A_recon.to(device).squeeze(0).detach().cpu().numpy()
                                    )

                                    # Slice only the [0:n, 0:n] parts for plotting
                                    im1 = axes[0, i].imshow(
                                        A_noisy_vis[:n, :n],
                                        cmap="viridis",
                                        interpolation="none",
                                    )
                                    axes[0, i].set_title(f"Noisy (ε={eps})")

                                    im2 = axes[1, i].imshow(
                                        A_recon_vis[:n, :n],
                                        cmap="viridis",
                                        interpolation="none",
                                    )
                                    axes[1, i].set_title("Reconstructed")

                            
                                # Log visualization to wandb if enabled
                                if j == 0:
                                    wandb.log(
                                        {
                                            "Test graph reconstruction": wandb.Image(
                                                fig
                                            )
                                        },
                                        step=epoch,
                                    )
                                else:
                                    wandb.log(
                                        {
                                            "Train graph reconstruction": wandb.Image(
                                                fig
                                            )
                                        },
                                        step=epoch,
                                    )
                                plt.close(fig)
                                plt.close('all')

                        else:
                            # visualize the results
                            eps_values = noise_levels
                            # Get reconstructions from the model for different noise levels
                            test_idx = np.random.randint(len(As_test))
                            train_idx = np.random.randint(len(As_train))
                            A_test = As_test[test_idx]
                            A_train = As_train[train_idx]

                            for j, A_orig in enumerate([A_test, A_train]):
                                A_orig = A_orig.unsqueeze(0)

                                # l_orig, V_orig = torch.linalg.eigh(A_orig)

                                fig_size = 4

                                # Create a figure with subplots - 2 rows for noisy/recon pairs, 5 columns for noise levels
                                fig, axes = plt.subplots(
                                    2,
                                    len(eps_values),
                                    figsize=(
                                        fig_size * len(eps_values),
                                        fig_size * 2,
                                    ),
                                )

                                # Plot pairs of noisy and reconstructed matrices for each noise level
                                for i, eps in enumerate(eps_values):
                                    A_noisy = add_digress_noise(A_orig, eps)[0]
                                    with torch.no_grad():
                                        A_recon = model(
                                            A_noisy.float().to(device)
                                        )
                                        if "digress" in model_type:
                                            A_recon = torch.nn.functional.sigmoid(A_recon)

                                        
                                    plt.figure()

                                    

                                    # Plot noisy matrix on top row
                                    im1 = axes[0, i].imshow(
                                        A_noisy.squeeze(0)
                                        .detach()
                                        .cpu()
                                        .numpy(),
                                        cmap="viridis",
                                    )
                                    axes[0, i].set_title(f"Noisy (ε={eps})")

                                    # Plot reconstructed matrix below
                                    im2 = axes[1, i].imshow(
                                        A_recon.squeeze(0)
                                        .detach()
                                        .cpu()
                                        .numpy(),
                                        cmap="viridis",
                                    )
                                    axes[1, i].set_title("Reconstructed")

                                
                                # Log visualization to wandb if enabled
                                if j == 0:
                                    wandb.log(
                                        {
                                            "Test graph reconstruction": wandb.Image(
                                                fig
                                            )
                                        },
                                        step=epoch,
                                    )
                                else:
                                    wandb.log(
                                        {
                                            "Train graph reconstruction": wandb.Image(
                                                fig
                                            )
                                        },
                                        step=epoch,
                                    )
                                plt.close(fig)
                                plt.close('all')

        # Save the trained model
        # torch.save(model.state_dict(), "models/denoiser_model_1.pt")
        # print("Model saved successfully to denoiser_model_1.pt")

        if use_wandb:
            wandb.finish()